<a href="https://colab.research.google.com/github/zaidachikzai-hub/Bus-118S/blob/main/Coding_Exercise_Prompt_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import json
from dataclasses import dataclass

@dataclass
class ChainResult:
    step1: dict
    step2: str
    step3: str
    step4: dict

def safe_json_loads(s: str) -> dict:
    try:
        return json.loads(s)
    except Exception as e:
        return {"error": f"Invalid JSON: {e}", "raw": s}

def mock_llm(system_prompt: str, user_prompt: str) -> str:
    text = user_prompt.lower()

    if "classify the customer message" in text:
        category = "login" if ("password" in text or "login" in text) else "billing" if ("charged" in text or "refund" in text or "invoice" in text) else "shipping" if ("delivery" in text or "tracking" in text) else "technical"
        severity = "high" if ("locked out" in text or "can't access" in text) else "medium"
        missing = []
        if category == "login":
            missing = ["email on the account", "error message shown", "when the issue started"]
        elif category == "billing":
            missing = ["order number", "date of charge", "last 4 digits of card used"]
        elif category == "shipping":
            missing = ["order number", "shipping address zip", "tracking number if available"]
        else:
            missing = ["device type", "app or website", "steps to reproduce"]

        return json.dumps({"category": category, "severity": severity, "missing_info": missing})

    if "write one message to the customer to collect the missing information" in text:
        start = user_prompt.find("You still need:")
        need_block = user_prompt[start:].splitlines()[0:2]
        need_line = need_block[1] if len(need_block) > 1 else ""
        return (
            "Thanks. I can help.\n"
            "1) What email is on the account\n"
            "2) What exact error message do you see\n"
            "3) When did this start"
        )

    if "propose the best next steps" in text:
        return (
            "1) Diagnosis\n"
            "This looks like an account login issue, likely a password reset or lockout.\n"
            "2) Steps customer should try\n"
            "Try a password reset. Check spam for the reset email. Try an incognito window. Confirm caps lock is off.\n"
            "3) What you will do next\n"
            "I will verify the account status and reset the lockout if needed once I have your email and the error message."
        )

    if "decide whether to escalate" in text:
        escalate = False
        reason = "No escalation rule triggered."
        routing = "tier_1_support"
        if "severity=high" in text or "security_risk=true" in text or "attempts=2" in text or "attempts=3" in text:
            escalate = True
            reason = "Escalation rule triggered."
            routing = "tier_2_or_security"
        return json.dumps({"escalate": escalate, "reason": reason, "routing_queue": routing})

    return "Unhandled prompt"

def run_prompt_chain(customer_message: str, customer_answers: str, attempts: int, security_risk: bool) -> ChainResult:
    system_1 = "You are a customer support triage assistant. Be concise. Output JSON only."
    user_1 = (
        "Classify the customer message into one category from:\n"
        "billing, login, shipping, returns, technical, other\n"
        "Also output severity from:\n"
        "low, medium, high\n"
        "Also output a list of missing_info you need next.\n"
        f"Customer message:\n{customer_message}"
    )
    step1_raw = mock_llm(system_1, user_1)
    step1 = safe_json_loads(step1_raw)

    system_2 = "You are a customer support agent. Be polite and efficient. Ask only for the missing items. No extra questions."
    user_2 = (
        f"You already classified the issue as:\n{step1.get('category')}\n"
        f"You still need:\n{step1.get('missing_info')}\n"
        "Write one message to the customer to collect the missing information.\n"
        "Constraints:\n"
        "Max 3 questions.\n"
        "Use numbered questions."
    )
    step2 = mock_llm(system_2, user_2)

    system_3 = "You are a customer support agent. Provide a practical solution. If you cannot solve without escalation, say so."
    user_3 = (
        "Use the data below to propose the best next steps.\n"
        f"Category:\n{step1.get('category')}\n"
        f"Severity:\n{step1.get('severity')}\n"
        f"Customer message:\n{customer_message}\n"
        f"Customer answers:\n{customer_answers}\n"
        "Output format:\n"
        "1) Diagnosis\n"
        "2) Steps customer should try\n"
        "3) What you will do next"
    )
    step3 = mock_llm(system_3, user_3)

    system_4 = "You are a support policy checker. Output JSON only."
    user_4 = (
        "Decide whether to escalate based on:\n"
        "Escalate if severity is high, or if account security risk is present, or if customer is blocked after 2 attempts.\n"
        f"Inputs:\ncategory={step1.get('category')}\nseverity={step1.get('severity')}\nattempts={attempts}\nsecurity_risk={str(security_risk).lower()}\n"
        "Output JSON keys:\n"
        "escalate, reason, routing_queue"
    )
    step4_raw = mock_llm(system_4, user_4)
    step4 = safe_json_loads(step4_raw)

    return ChainResult(step1=step1, step2=step2, step3=step3, step4=step4)

customer_message = "I cannot login. The site says my account is locked after I tried resetting my password."
customer_answers = "Email: zaid@example.com. Error: Account locked. Started today after 3 failed attempts."
result = run_prompt_chain(customer_message, customer_answers, attempts=2, security_risk=False)

print("STEP 1 JSON")
print(json.dumps(result.step1, indent=2))
print("\nSTEP 2 QUESTION MESSAGE")
print(result.step2)
print("\nSTEP 3 SOLUTION")
print(result.step3)
print("\nSTEP 4 ESCALATION JSON")
print(json.dumps(result.step4, indent=2))

STEP 1 JSON
{
  "category": "login",
  "severity": "medium",
  "missing_info": [
    "email on the account",
    "error message shown",
    "when the issue started"
  ]
}

STEP 2 QUESTION MESSAGE
Thanks. I can help.
1) What email is on the account
2) What exact error message do you see
3) When did this start

STEP 3 SOLUTION
1) Diagnosis
This looks like an account login issue, likely a password reset or lockout.
2) Steps customer should try
Try a password reset. Check spam for the reset email. Try an incognito window. Confirm caps lock is off.
3) What you will do next
I will verify the account status and reset the lockout if needed once I have your email and the error message.

STEP 4 ESCALATION JSON
{
  "escalate": true,
  "reason": "Escalation rule triggered.",
  "routing_queue": "tier_2_or_security"
}


In [13]:
import re
import pandas as pd

REACT_PROMPT = """System:
You are a senior Python engineer. You follow a ReACT loop. You must separate stages with these exact headers:
PLAN
CODE
RUN_INSTRUCTIONS
OBSERVATIONS
FIX
Rules:
In CODE, output only Python code in one code block.
In RUN_INSTRUCTIONS, tell me exactly what to run.
In OBSERVATIONS, explain what to check in the output.
In FIX, if errors occur, propose edits.

User:
Task:
Write Python code that loads a small in memory dataset of customer tickets, cleans it, computes:
1) Count of tickets by category
2) Average resolution_minutes by category
3) Top 3 most common words in the issue_text after basic cleaning
Constraints:
Use only pandas and re from the standard library.
Include basic error handling for missing columns.
Print outputs clearly.
"""

print("REACT PROMPT USED")
print(REACT_PROMPT)

def basic_clean(text: str) -> list:
    text = (text or "").lower()
    text = re.sub(r"[^a-z0-9\\s]", " ", text)
    tokens = [t for t in text.split() if len(t) >= 3]
    stop = {"the", "and", "for", "with", "that", "this", "you", "your", "are", "was", "not"}
    return [t for t in tokens if t not in stop]

def run_task():
    df = pd.DataFrame(
        [
            {"ticket_id": 1, "category": "login", "issue_text": "Password reset email not arriving", "resolution_minutes": 25},
            {"ticket_id": 2, "category": "billing", "issue_text": "Charged twice for one order", "resolution_minutes": 40},
            {"ticket_id": 3, "category": "shipping", "issue_text": "Tracking shows delivered but I did not receive it", "resolution_minutes": 60},
            {"ticket_id": 4, "category": "login", "issue_text": "Account locked after too many attempts", "resolution_minutes": 35},
            {"ticket_id": 5, "category": "billing", "issue_text": "Need refund for canceled subscription", "resolution_minutes": 50},
        ]
    )

    required = {"category", "issue_text", "resolution_minutes"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    df["category"] = df["category"].astype(str).str.strip().str.lower()
    df["resolution_minutes"] = pd.to_numeric(df["resolution_minutes"], errors="coerce")
    df = df.dropna(subset=["resolution_minutes"])

    print("1) Ticket count by category")
    print(df["category"].value_counts())

    print("\n2) Average resolution_minutes by category")
    print(df.groupby("category")["resolution_minutes"].mean().round(2))

    all_tokens = []
    for t in df["issue_text"].astype(str).tolist():
        all_tokens.extend(basic_clean(t))

    freq = pd.Series(all_tokens).value_counts().head(3)
    print("\n3) Top 3 words in issue_text")
    print(freq)

run_task()

REACT PROMPT USED
System:
You are a senior Python engineer. You follow a ReACT loop. You must separate stages with these exact headers:
PLAN
CODE
RUN_INSTRUCTIONS
OBSERVATIONS
FIX
Rules:
In CODE, output only Python code in one code block.
In RUN_INSTRUCTIONS, tell me exactly what to run.
In OBSERVATIONS, explain what to check in the output.
In FIX, if errors occur, propose edits.

User:
Task:
Write Python code that loads a small in memory dataset of customer tickets, cleans it, computes:
1) Count of tickets by category
2) Average resolution_minutes by category
3) Top 3 most common words in the issue_text after basic cleaning
Constraints:
Use only pandas and re from the standard library.
Include basic error handling for missing columns.
Print outputs clearly.

1) Ticket count by category
category
login       2
billing     2
shipping    1
Name: count, dtype: int64

2) Average resolution_minutes by category
category
billing     45.0
login       30.0
shipping    60.0
Name: resolution_minut

In [14]:
ORIGINAL_TEXT = (
    "Our support team handled 500 tickets last week. Login issues were the top category, followed by billing. "
    "Average resolution time improved from 52 minutes to 41 minutes after we updated our internal playbook. "
    "Customers still reported frustration with password reset emails not arriving quickly. "
    "We plan to add clearer status messaging and a faster email provider."
)

summary_prompt = "Summarize the text in 6 to 8 sentences."
print("ORIGINAL SUMMARY PROMPT")
print(summary_prompt)

before_summary = (
    "The team handled many support tickets last week. Login and billing issues were common. "
    "Resolution time improved after a playbook update. Customers felt frustrated about password reset emails. "
    "The team will improve messaging and email speed. The changes should help customer satisfaction."
)
print("\nBEFORE SUMMARY")
print(before_summary)

critique_and_revise_prompt = (
    "Critique the summary against these criteria:\n"
    "Accuracy. No new facts.\n"
    "Clarity. Simple wording.\n"
    "Audience. General reader.\n"
    "Length. 6 to 8 sentences.\n"
    "Structure. First sentence states the main point. Remaining sentences add key details.\n"
    "Then produce a revised summary that satisfies all criteria.\n"
    "Output format:\n"
    "CRITIQUE:\n"
    "REVISED SUMMARY:"
)
print("\nSELF CRITIQUE PROMPT")
print(critique_and_revise_prompt)

critique = (
    "CRITIQUE:\n"
    "The first sentence is vague and does not state the main point. It also misses the ticket count. "
    "It does not mention the exact resolution time change. It is only 6 sentences but some are generic.\n"
)
revised = (
    "REVISED SUMMARY:\n"
    "The support team processed 500 tickets last week and improved average resolution time. "
    "Login issues were the most common category, with billing next. "
    "Average resolution time dropped from 52 minutes to 41 minutes after an internal playbook update. "
    "Customers still reported frustration because password reset emails did not arrive quickly. "
    "The team plans to add clearer status messaging during resets. "
    "The team also plans to switch to a faster email provider to reduce delays."
)

print("\nCRITIQUE AND REVISED SUMMARY OUTPUT")
print(critique + revised)

ORIGINAL SUMMARY PROMPT
Summarize the text in 6 to 8 sentences.

BEFORE SUMMARY
The team handled many support tickets last week. Login and billing issues were common. Resolution time improved after a playbook update. Customers felt frustrated about password reset emails. The team will improve messaging and email speed. The changes should help customer satisfaction.

SELF CRITIQUE PROMPT
Critique the summary against these criteria:
Accuracy. No new facts.
Clarity. Simple wording.
Audience. General reader.
Length. 6 to 8 sentences.
Structure. First sentence states the main point. Remaining sentences add key details.
Then produce a revised summary that satisfies all criteria.
Output format:
CRITIQUE:
REVISED SUMMARY:

CRITIQUE AND REVISED SUMMARY OUTPUT
CRITIQUE:
The first sentence is vague and does not state the main point. It also misses the ticket count. It does not mention the exact resolution time change. It is only 6 sentences but some are generic.
REVISED SUMMARY:
The support team 